# 04 - Experiment analysis: reading the A/B/C/D results

**What:** turns the runs/ directory into the comparison table and the four layman charts.

**How to read each chart:**
- 01_which_recipe_learned_best: LOWER bar = better learner.
- 02_speed_and_memory: HIGHER tok/s = faster; LOWER MB = cheaper.
- 03_learning_speed: SHORTER bar = reached the goal with less reading.
- 04_verdict_card: the one-glance poster.

In [ ]:
# --- locate the repo (works locally AND on Colab/Kaggle pasted into a
# fresh notebook: it finds an existing clone or clones from GitHub) ------
import os, sys, subprocess

def _find_repo():
    here = os.path.abspath("")
    candidates = [here, os.path.dirname(here),
                  os.path.join(here, "nano-gpt-lab"),
                  "/kaggle/working/nano-gpt-lab",
                  "/content/nano-gpt-lab"]
    for d in candidates:
        if d and os.path.exists(os.path.join(d, "scripts", "train.py")):
            return d
    return None

repo = _find_repo()
if repo is None:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        dest = "/kaggle/working/nano-gpt-lab"
    elif "google.colab" in sys.modules:
        dest = "/content/nano-gpt-lab"
    else:
        dest = os.path.join(os.path.abspath(""), "nano-gpt-lab")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Th3Samaritan/nano-gpt-lab.git",
                    dest], check=True)
    repo = dest
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # Windows OMP clash
sys.path.insert(0, repo)
os.chdir(repo)
print("repo on path:", repo)

In [ ]:
from src.evaluation.analysis import analyze
report = analyze("shakespeare")          # add size="gpt2" for T4 runs
print(report["table"])

In [ ]:
from IPython.display import Image, display
for name, path in sorted(report["plots"].items()):
    print(name)
    display(Image(path))

In [ ]:
# --- the deeper questions (plan section 18) ------------------------------
import csv, os, math
from src.evaluation.analysis import collect_summaries

summaries = collect_summaries("shakespeare")
for s in sorted(summaries, key=lambda d: d["best_val_loss"]):
    print(f"{s['attention']:>12} + {s['position']:<8} "
          f"seed {s['seed']:>3} | best val {s['best_val_loss']:.4f} "
          f"({s['val_ppl']:,.1f} choices) | {s['tokens_per_sec']:>8,.0f} tok/s "
          f"| {s['peak_gpu_mem_mb']:>6,.0f} MB")
print()
print("Ask, per pair of rows:")
print("  * vanilla -> flash (A vs C, B vs D): does loss change? It should")
print("    NOT - flash is a memory/speed trick, not a quality trick.")
print("  * learned -> rope (A vs B, C vs D): does quality change at equal")
print("    tokens? Check 03_learning_speed for the 'fewer tokens' claim.")

**Report template (plan section 38):** hypothesis, setup, result, why you expected it, what happened, limitation, next experiment. Write one of these per study - that is the actual research.